# Evaluation of models on medical data

This notebook combines five-fold cross-validation results from tuning with a single final-refit result on the held-out test set.

The reporting unit is the final model linked to its source Optuna trial.

## Library imports and configuration

Configuration of data sources and shared reporting style.

In [ ]:
import json
import os
from itertools import combinations
from pathlib import Path

import dotenv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from my_project.reporting import (
    CATEGORICAL_PALETTE,
    configure_notebook_plots,
    graph_display_name,
    save_figure,
    save_table,
)

dotenv.load_dotenv(override=True)

In [ ]:
EXPECTED_FOLDS = 5
PALETTE = CATEGORICAL_PALETTE
GRAPH_ARTIFACTS_FILENAME = "mlflow_graph_artifacts.json"

RESULTS_DIR = os.getenv("RESULTS_DIR")
if not RESULTS_DIR:
    raise ValueError("RESULTS_DIR must be configured")

configure_notebook_plots()
pd.set_option("display.max_columns", None)

## Loading MLflow results

The notebook loads previously prepared run and artifact snapshots from `RESULTS_DIR`. Both files are created by `mlflow_results_evaluate_synthetic_select_data.ipynb`, keeping the evaluation stage independent of MLflow.

In [ ]:
csv_path = os.path.join(RESULTS_DIR, "mlflow_selected_runs_df.csv")
runs_df = pd.read_csv(csv_path, dtype=str)
runs_df = runs_df[runs_df["experiment_name"] == "medical"]

graph_artifacts_path = os.path.join(RESULTS_DIR, GRAPH_ARTIFACTS_FILENAME)
with open(graph_artifacts_path, encoding="utf-8") as artifact_file:
    graph_artifact_data = json.load(artifact_file)
graph_artifact_lookup = {
    (str(artifact["run_id"]), artifact["artifact_path"]): artifact
    for artifact in graph_artifact_data["artifacts"]
}

In [ ]:
model_cls_raw = runs_df["model_cls"]
source_trial_id = runs_df[["trial_id", "source_trial_number"]].bfill(axis=1).iloc[:, 0]

runs_df = runs_df.assign(
    model=model_cls_raw.astype("string").str.removeprefix("My"),
    model_cls_raw=model_cls_raw,
    graph=runs_df["graph_name"],
    trial_id=pd.to_numeric(source_trial_id, errors="coerce").astype("Int64"),
    fold_id=pd.to_numeric(runs_df["fold_id"], errors="coerce").astype("Int64"),
    n_edges=pd.to_numeric(runs_df["n_edges"], errors="coerce").astype("Int64"),
    val_auc=pd.to_numeric(runs_df["val/auc"], errors="coerce"),
    val_avg_precision=pd.to_numeric(runs_df["val/avg_precision"], errors="coerce"),
    test_auc=pd.to_numeric(runs_df["test/auc"], errors="coerce"),
    test_avg_precision=pd.to_numeric(runs_df["test/avg_precision"], errors="coerce"),
)

## Separating tuning and final-fit runs

`fold` runs contain cross-validation results. `final_refit` runs contain a single evaluation on the held-out test set.

In [ ]:
tuning_folds = runs_df[runs_df["run_type"].eq("fold")].copy()
final_runs = runs_df[runs_df["run_type"].eq("final_refit")].copy()

pd.Series(
    {
        "tuning folds": len(tuning_folds),
        "final refits": len(final_runs),
    }
)

## Selecting the best CV trials and joining final fits

For each configuration, the trial with the highest mean cross-validation PR AUC is selected. Its final-refit result is joined when available.

In [ ]:
JOIN_KEYS = ["experiment_id", "model_cls_raw", "graph", "trial_id"]
CONFIG_KEYS = ["experiment_id", "model_cls_raw", "graph"]
final_keys = final_runs[JOIN_KEYS + ["model", "n_edges", "source_study_name"]].copy()

cv_trial_scores = (
    tuning_folds.groupby(JOIN_KEYS, dropna=False)["val_avg_precision"]
    .mean()
    .rename("cv_avg_precision_mean")
    .reset_index()
)
best_cv_keys = cv_trial_scores.loc[
    cv_trial_scores.groupby(CONFIG_KEYS, dropna=False)[
        "cv_avg_precision_mean"
    ].idxmax(),
    JOIN_KEYS,
]
linked_folds = tuning_folds.merge(
    best_cv_keys,
    on=JOIN_KEYS,
    how="inner",
    validate="many_to_one",
)

pd.Series(
    {
        "best CV models": len(best_cv_keys),
        "final models": len(final_keys),
        "linked folds": len(linked_folds),
    }
)

## Join completeness validation

Each selected model must have exactly five distinct folds with IDs 0-4. The final refit is optional, but when present, it must reference a model found in the CV results.

In [ ]:
coverage = (
    linked_folds.groupby(JOIN_KEYS, dropna=False)
    .agg(
        n_rows=("fold_id", "size"),
        n_folds=("fold_id", "nunique"),
        fold_ids=("fold_id", lambda values: tuple(sorted(values.dropna().astype(int)))),
    )
    .reset_index()
)

expected_fold_ids = tuple(range(EXPECTED_FOLDS))
invalid_coverage = coverage[
    (coverage["n_rows"] != EXPECTED_FOLDS)
    | (coverage["n_folds"] != EXPECTED_FOLDS)
    | coverage["fold_ids"].apply(lambda fold_ids: fold_ids != expected_fold_ids)
]
final_without_cv = final_keys.merge(
    coverage[JOIN_KEYS], on=JOIN_KEYS, how="left", indicator=True
).query("_merge != 'both'")

assert final_keys[JOIN_KEYS].duplicated().sum() == 0, "Duplicate final source keys"
assert invalid_coverage.empty, f"Incomplete CV folds:\n{invalid_coverage}"
assert final_without_cv.empty, f"Final runs without tuning folds:\n{final_without_cv}"
coverage

## Aggregating CV and test results

For the best trial of each configuration, the mean and standard deviation of metrics across five folds are calculated. The single test-set result is joined when a final refit exists.

In [ ]:
cv_summary = (
    linked_folds.groupby(JOIN_KEYS + ["model", "n_edges"], dropna=False)
    .agg(
        n_folds=("fold_id", "nunique"),
        cv_auc_mean=("val_auc", "mean"),
        cv_auc_std=("val_auc", "std"),
        cv_avg_precision_mean=("val_avg_precision", "mean"),
        cv_avg_precision_std=("val_avg_precision", "std"),
    )
    .reset_index()
)
final_test = final_runs[JOIN_KEYS + ["test_auc", "test_avg_precision"]]
results = cv_summary.merge(
    final_test,
    on=JOIN_KEYS,
    how="left",
    validate="one_to_one",
)
assert len(results) == len(cv_summary)
results

## Creating the results table

The table places cross-validation and single test-set evaluations side by side. Rows are sorted by mean cross-validation PR AUC.

In [ ]:
GRAPHLESS_MODELS = {"MLP", "XGBoost"}


def make_model_label(row):
    model = str(row["model"])
    if model in GRAPHLESS_MODELS:
        return model
    graph = str(row["graph"])
    graph_label = graph_display_name(graph)
    if "_permuted_" in graph:
        graph_label = f"{graph_label} permuted {graph.rsplit('_permuted_', 1)[1]}"
    return f"{model} / {graph_label} / {int(row['n_edges'])}"


result_columns = [
    "model",
    "graph",
    "n_edges",
    "trial_id",
    "n_folds",
    "cv_auc_mean",
    "cv_auc_std",
    "cv_avg_precision_mean",
    "cv_avg_precision_std",
    "test_auc",
    "test_avg_precision",
]
results_table = results.sort_values(
    "cv_avg_precision_mean", ascending=False
).reset_index(drop=True)
results_table["label"] = results_table.apply(make_model_label, axis=1)
results_table = results_table[result_columns + ["label"]]
results_table

## Results visualization

Each metric has a separate plot consistent with the synthetic-analysis style. Colored box plots show the distribution across five folds, small black crosses indicate fold means, and hollow black diamonds indicate available final-model results on the held-out test set.

In [ ]:
metric_columns = [
    "cv_auc_mean",
    "cv_auc_std",
    "test_auc",
    "cv_avg_precision_mean",
    "cv_avg_precision_std",
    "test_avg_precision",
]
permuted_mask = results_table["graph"].str.contains("_permuted_", na=False)

non_permuted_results = results_table.loc[
    ~permuted_mask,
    ["model", "graph", "n_edges", *metric_columns],
].copy()

permuted_folds = linked_folds.loc[
    linked_folds["graph"].str.contains("_permuted_", na=False)
].copy()
permuted_cv = permuted_folds.groupby(["model", "n_edges"], as_index=False).agg(
    cv_auc_mean=("val_auc", "mean"),
    cv_auc_std=("val_auc", "std"),
    cv_avg_precision_mean=("val_avg_precision", "mean"),
    cv_avg_precision_std=("val_avg_precision", "std"),
)
permuted_test = (
    results_table.loc[permuted_mask]
    .groupby(["model", "n_edges"], as_index=False)
    .agg(
        test_auc=("test_auc", "mean"),
        test_avg_precision=("test_avg_precision", "mean"),
    )
)
permuted_results = permuted_cv.merge(
    permuted_test,
    on=["model", "n_edges"],
    how="left",
    validate="one_to_one",
)
permuted_results["graph"] = "ii_perm"

export_result_df = pd.concat(
    [non_permuted_results, permuted_results],
    ignore_index=True,
)[["model", "graph", "n_edges", *metric_columns]]
ii_mask = export_result_df["graph"].str.fullmatch(r"ii_graph_\d+", na=False)
graphless_mask = export_result_df["model"].isin(GRAPHLESS_MODELS)
export_result_df.loc[ii_mask, "graph"] = "ii"
export_result_df.loc[graphless_mask, ["graph", "n_edges"]] = pd.NA
export_result_df.loc[export_result_df["n_edges"] == -1, "n_edges"] = pd.NA

export_result_df["test_auc_rank"] = (
    export_result_df["test_auc"].rank(method="dense", ascending=False).astype("Int64")
)
export_result_df["test_avg_precision_rank"] = (
    export_result_df["test_avg_precision"]
    .rank(method="dense", ascending=False)
    .astype("Int64")
)
common_columns = ["model", "graph", "n_edges"]
export_auc_table = (
    export_result_df[
        common_columns + ["cv_auc_mean", "cv_auc_std", "test_auc", "test_auc_rank"]
    ]
    .sort_values("test_auc", ascending=False, na_position="last")
    .reset_index(drop=True)
)
export_avg_precision_table = (
    export_result_df[
        common_columns
        + [
            "cv_avg_precision_mean",
            "cv_avg_precision_std",
            "test_avg_precision",
            "test_avg_precision_rank",
        ]
    ]
    .sort_values("test_avg_precision", ascending=False, na_position="last")
    .reset_index(drop=True)
)

save_table(
    export_auc_table.round(4),
    "evaluation__medical__cv_vs_final_test__auc",
    index=False,
)
save_table(
    export_avg_precision_table.round(4),
    "evaluation__medical__cv_vs_final_test__avg_precision",
    index=False,
)
display(export_auc_table)
display(export_avg_precision_table)

In [ ]:
ii_results = results_table.loc[
    results_table["graph"].str.startswith("ii_graph_", na=False),
    ["graph", "n_edges", *metric_columns],
].copy()
ii_results["graph"] = ii_results["graph"].map(
    lambda graph: (
        f"ii permuted {graph.rsplit('_permuted_', 1)[1]}"
        if "_permuted_" in graph
        else "ii"
    )
)
ii_results["test_auc_rank"] = (
    ii_results.groupby("n_edges")["test_auc"]
    .rank(method="dense", ascending=False)
    .astype("Int64")
)
ii_results["test_avg_precision_rank"] = (
    ii_results.groupby("n_edges")["test_avg_precision"]
    .rank(method="dense", ascending=False)
    .astype("Int64")
)

edge_counts = sorted(ii_results["n_edges"].dropna().astype(int).unique())
permutation_ids = sorted(
    int(graph.rsplit(" ", 1)[1])
    for graph in ii_results["graph"].unique()
    if graph.startswith("ii permuted ")
)
graph_order = ["ii", *(f"ii permuted {index}" for index in permutation_ids)]


def make_ii_metric_pivot(df, *, cv_mean, cv_std, test, rank):
    source_columns = [cv_mean, cv_std, test, rank]
    statistic_labels = {
        cv_mean: "CV mean",
        cv_std: "CV std",
        test: "test",
        rank: "rank",
    }
    table = df.pivot(
        index="graph",
        columns="n_edges",
        values=source_columns,
    ).swaplevel(0, 1, axis=1)
    expected_columns = pd.MultiIndex.from_product(
        [edge_counts, source_columns],
        names=["n_edges", "statistic"],
    )
    table = table.reindex(index=graph_order, columns=expected_columns)
    table.columns = pd.MultiIndex.from_tuples(
        [
            (n_edges, statistic_labels[statistic])
            for n_edges, statistic in table.columns
        ],
        names=["n_edges", "statistic"],
    )
    column_types = {
        column: "Int64" if column[-1] == "rank" else "float64"
        for column in table.columns
    }
    return table.astype(column_types)


ii_auc_table = make_ii_metric_pivot(
    ii_results,
    cv_mean="cv_auc_mean",
    cv_std="cv_auc_std",
    test="test_auc",
    rank="test_auc_rank",
)
ii_avg_precision_table = make_ii_metric_pivot(
    ii_results,
    cv_mean="cv_avg_precision_mean",
    cv_std="cv_avg_precision_std",
    test="test_avg_precision",
    rank="test_avg_precision_rank",
)

save_table(
    ii_auc_table.round(4),
    "evaluation__medical__ii_permutations_cv_vs_final_test__auc",
)
save_table(
    ii_avg_precision_table.round(4),
    "evaluation__medical__ii_permutations_cv_vs_final_test__avg_precision",
)
display(ii_auc_table)
display(ii_avg_precision_table)

In [ ]:
def make_ii_variant_label(graph):
    graph = str(graph)
    graph_label = graph_display_name(graph)
    if "_permuted_" in graph:
        permutation_id = graph.rsplit("_permuted_", 1)[1]
        return f"{graph_label} permuted {permutation_id}"
    return graph_label


def plot_ii_facets_by_n_edges(
    folds_df,
    results_df,
    metric_config,
    *,
    sort_by="cv_mean",
):
    if sort_by not in {"cv_mean", "test"}:
        raise ValueError("sort_by must be 'cv_mean' or 'test'")

    graph_mask = folds_df["graph"].str.startswith("ii_graph_", na=False)
    plot_folds = folds_df.loc[graph_mask].copy()
    plot_results = results_df.loc[
        results_df["graph"].str.startswith("ii_graph_", na=False)
    ].copy()

    for frame in (plot_folds, plot_results):
        frame["graph_variant"] = frame["graph"].map(make_ii_variant_label)

    variant_order = (
        plot_folds[["graph", "graph_variant"]]
        .sort_values("graph")["graph_variant"]
        .drop_duplicates()
        .tolist()
    )
    variant_palette = dict(
        zip(variant_order, sns.color_palette(PALETTE, n_colors=len(variant_order)))
    )
    edge_counts = sorted(plot_folds["n_edges"].dropna().astype(int).unique())
    middle_facet = (len(edge_counts) - 1) // 2
    figure, axes = plt.subplots(
        len(edge_counts),
        1,
        figsize=(10, 3 * len(edge_counts)),
        sharex=False,
        sharey=True,
        squeeze=False,
    )

    for facet_index, (axis, n_edges) in enumerate(zip(axes.flat, edge_counts)):
        facet_folds = plot_folds[plot_folds["n_edges"] == n_edges]
        facet_results = plot_results[plot_results["n_edges"] == n_edges]
        cv_means = facet_folds.groupby("graph_variant")[metric_config["cv"]].mean()
        test_values = facet_results.groupby("graph_variant")[
            metric_config["test"]
        ].mean()
        sort_values = cv_means if sort_by == "cv_mean" else test_values
        order = sort_values.sort_values(
            ascending=False,
            na_position="last",
        ).index.tolist()
        positions = np.arange(len(order))

        sns.boxplot(
            data=facet_folds,
            x="graph_variant",
            y=metric_config["cv"],
            hue="graph_variant",
            order=order,
            hue_order=order,
            palette=variant_palette,
            legend=False,
            width=0.65,
            fliersize=3,
            linewidth=1,
            ax=axis,
        )
        axis.scatter(
            positions,
            cv_means.reindex(order),
            marker="x",
            color="black",
            s=25,
            linewidths=0.6,
            zorder=11,
        )
        axis.scatter(
            positions,
            test_values.reindex(order),
            marker="D",
            facecolors="white",
            edgecolors="black",
            s=42,
            linewidths=1.1,
            zorder=9,
        )
        axis.set_xlabel("")
        axis.set_ylabel(metric_config["y_label"] if facet_index == middle_facet else "")
        axis.tick_params(axis="x", labelrotation=90)

        row_label = str(n_edges)
        if facet_index == middle_facet:
            row_label = f"liczba krawędzi\n\n{row_label}"
        axis.text(
            1.01,
            0.5,
            row_label,
            transform=axis.transAxes,
            rotation=270,
            ha="left",
            va="center",
            multialignment="center",
        )

    axes.flat[-1].set_xlabel("wariant grafu")
    axes.flat[0].scatter(
        [],
        [],
        marker="x",
        color="black",
        s=25,
        linewidths=0.6,
        label="średnia z walidacji krzyżowej",
    )
    axes.flat[0].scatter(
        [],
        [],
        marker="D",
        facecolors="white",
        edgecolors="black",
        s=42,
        linewidths=1.1,
        label="wynik na zbiorze testowym",
    )
    handles, labels = axes.flat[0].get_legend_handles_labels()
    figure.legend(
        handles,
        labels,
        loc="upper center",
        ncol=2,
        frameon=False,
    )
    figure.tight_layout(rect=(0, 0, 0.95, 0.96))
    return figure

In [ ]:
def make_plot_label(row, group_permuted):
    if group_permuted and "_permuted_" in str(row["graph"]):
        return f"{row['model']} / ii_perm / {int(row['n_edges'])}"
    return make_model_label(row)


def plot_cv_vs_test(
    folds_df,
    results_df,
    metric_config,
    *,
    group_permuted=False,
    font_size=12,
):
    plot_folds = folds_df.copy()
    plot_folds["label"] = plot_folds.apply(
        make_plot_label, axis=1, group_permuted=group_permuted
    )

    plot_results = results_df.copy()
    plot_results["label"] = plot_results.apply(
        make_plot_label, axis=1, group_permuted=group_permuted
    )
    test_values = (
        plot_results.groupby("label")[metric_config["test"]]
        .mean()
        .sort_values(ascending=False, na_position="last")
    )
    order = test_values.index.tolist()

    positions = np.arange(len(order))
    figure, axis = plt.subplots(figsize=(11, 8))
    sns.boxplot(
        data=plot_folds,
        x="label",
        y=metric_config["cv"],
        hue="label",
        order=order,
        hue_order=order,
        palette=PALETTE,
        legend=False,
        width=0.8,
        fliersize=3,
        linewidth=1,
        ax=axis,
    )
    axis.scatter(
        positions,
        test_values.reindex(order),
        marker="D",
        facecolors="white",
        edgecolors="black",
        s=42,
        linewidths=1.1,
        label="wynik na zbiorze testowym",
        zorder=9,
    )
    sns.pointplot(
        data=plot_folds,
        x="label",
        y=metric_config["cv"],
        hue="label",
        order=order,
        hue_order=order,
        estimator=np.mean,
        errorbar=None,
        dodge=0,
        markers="x",
        linestyles="none",
        markersize=5,
        markeredgewidth=0.6,
        palette={label: "black" for label in order},
        legend=False,
        zorder=11,
        ax=axis,
    )
    axis.scatter(
        [],
        [],
        marker="x",
        color="black",
        s=25,
        linewidths=0.6,
        label="średnia z walidacji krzyżowej",
    )
    axis.set_xlabel("klasa modelu / nazwa grafu / liczba krawędzi", fontsize=font_size)
    axis.set_ylabel(metric_config["y_label"], fontsize=font_size)
    axis.tick_params(axis="x", labelrotation=90, labelsize=font_size)
    handles, labels = axis.get_legend_handles_labels()
    axis.legend(
        handles[::-1],
        labels[::-1],
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=2,
        frameon=False,
        fontsize=font_size,
    )
    figure.tight_layout()
    return figure

In [ ]:
PLOT_METRICS = {
    "auc": {
        "y_label": "cv_val/auc",
        "cv": "val_auc",
        "test": "test_auc",
    },
    "avg_precision": {
        "y_label": "cv_val/pr_auc",
        "cv": "val_avg_precision",
        "test": "test_avg_precision",
    },
}

In [ ]:
for metric_key, metric_config in PLOT_METRICS.items():
    figure = plot_cv_vs_test(
        linked_folds,
        results_table,
        metric_config,
        group_permuted=True,
    )
    save_figure(
        figure,
        f"evaluation__medical__cv_vs_final_test__{metric_key}__grouped_permuted",
    )
    plt.show()

In [ ]:
II_FACET_SORT_BY = "test"

for metric_key, metric_config in PLOT_METRICS.items():
    figure = plot_ii_facets_by_n_edges(
        linked_folds,
        results_table,
        metric_config,
        sort_by=II_FACET_SORT_BY,
    )
    save_figure(
        figure,
        f"evaluation__medical__cv_vs_final_test__{metric_key}__ii_facets_by_n_edges",
    )
    plt.show()

In [ ]:
for metric_key, metric_config in PLOT_METRICS.items():
    figure = plot_cv_vs_test(
        linked_folds,
        results_table,
        metric_config,
        group_permuted=False,
    )
    save_figure(
        figure,
        f"evaluation__medical__cv_vs_final_test__{metric_key}",
    )
    plt.show()

## Interaction-graph edge stability

For the winning trial of each graph, the complete edge lists from `graph/estimated_graph.json` are compared across the five fold runs. These artifacts were previously stored in the local `mlflow_graph_artifacts.json` snapshot. The `graph_edges` field is not used because longer lists were truncated by the MLflow parameter-length limit.

If a final GNN refit exists, its complete structure from `final_graph.json` is additionally compared with the fold graphs. The CV report and edge-frequency plot also include configurations without a final refit, including permuted graphs.

Each edge in an available final graph also receives a stability rank within its graph variant. Rank is determined in descending order by the number of folds containing the edge; edges with equal support receive the same rank.

In [ ]:
CV_GRAPH_ARTIFACT_PATH = "graph/estimated_graph.json"
FINAL_GRAPH_ARTIFACT_PATH = "final_graph.json"


def load_edge_set(run_id, artifact_path):
    key = (str(run_id), artifact_path)
    try:
        payload = graph_artifact_lookup[key]
    except KeyError as error:
        raise FileNotFoundError(
            f"Graph artifact {artifact_path!r} for run {run_id} is missing from "
            f"{graph_artifacts_path}. Run the data-selection notebook first."
        ) from error

    edges = {
        tuple(sorted((str(source), str(target)))) for source, target in payload["edges"]
    }
    if len(edges) != payload["n_edges"]:
        raise ValueError(
            f"Invalid graph artifact for run {run_id}: "
            f"expected {payload['n_edges']} edges, found {len(edges)}"
        )
    return edges

In [ ]:
interaction_folds = linked_folds[
    linked_folds["graph"].str.startswith("ii_graph_", na=False)
].copy()
interaction_final_runs = final_runs[
    final_runs["graph"].str.startswith("ii_graph_", na=False)
].copy()
edge_sets_by_graph = {}
edge_presence_records = []

for graph_name, graph_folds in interaction_folds.groupby("graph", sort=True):
    fold_edge_sets = {}
    for row in graph_folds.sort_values("fold_id").itertuples():
        edge_set = load_edge_set(row.run_id, CV_GRAPH_ARTIFACT_PATH)
        fold_id = int(row.fold_id)
        fold_edge_sets[fold_id] = edge_set
        edge_presence_records.extend(
            {
                "graph": graph_name,
                "fold_id": fold_id,
                "source": source,
                "target": target,
            }
            for source, target in edge_set
        )
    if len(fold_edge_sets) != EXPECTED_FOLDS:
        raise ValueError(
            f"Expected {EXPECTED_FOLDS} graph artifacts for {graph_name}, "
            f"found {len(fold_edge_sets)}"
        )
    edge_sets_by_graph[graph_name] = fold_edge_sets

edge_frequency = (
    pd.DataFrame(edge_presence_records)
    .groupby(["graph", "source", "target"], as_index=False)
    .agg(n_folds=("fold_id", "nunique"))
)
edge_frequency["fold_fraction"] = edge_frequency["n_folds"] / EXPECTED_FOLDS

stability_records = []
final_edge_support_records = []
for graph_name, fold_edge_sets in edge_sets_by_graph.items():
    edge_sets = list(fold_edge_sets.values())
    pairwise_jaccard = [
        len(left & right) / len(left | right)
        for left, right in combinations(edge_sets, 2)
    ]
    common_edges = set.intersection(*edge_sets)
    union_edges = set.union(*edge_sets)
    stability_record = {
        "graph": graph_name,
        "edges_per_fold": len(edge_sets[0]),
        "final_edges": pd.NA,
        "common_cv_edges": len(common_edges),
        "union_cv_edges": len(union_edges),
        "common_cv_edge_fraction": len(common_edges) / len(edge_sets[0]),
        "mean_cv_pairwise_jaccard": np.mean(pairwise_jaccard),
        "min_cv_pairwise_jaccard": np.min(pairwise_jaccard),
        "mean_final_vs_fold_jaccard": np.nan,
        "min_final_vs_fold_jaccard": np.nan,
        "max_final_vs_fold_jaccard": np.nan,
        "final_edges_in_cv_union_fraction": np.nan,
        "final_edges_in_all_folds_fraction": np.nan,
    }

    final_matches = interaction_final_runs[
        interaction_final_runs["graph"] == graph_name
    ]
    if len(final_matches) > 1:
        raise ValueError(
            f"Expected at most one final run for {graph_name}, found {len(final_matches)}"
        )
    if len(final_matches) == 1:
        final_run = final_matches.iloc[0]
        final_edge_set = load_edge_set(
            final_run["run_id"],
            FINAL_GRAPH_ARTIFACT_PATH,
        )
        final_vs_fold_jaccard = [
            len(final_edge_set & fold_edges) / len(final_edge_set | fold_edges)
            for fold_edges in edge_sets
        ]
        final_edge_support_records.extend(
            {
                "graph": graph_name,
                "source": source,
                "target": target,
                "n_folds": sum(edge in fold_edges for fold_edges in edge_sets),
            }
            for edge in final_edge_set
            for source, target in [edge]
        )
        stability_record.update(
            {
                "final_edges": len(final_edge_set),
                "mean_final_vs_fold_jaccard": np.mean(final_vs_fold_jaccard),
                "min_final_vs_fold_jaccard": np.min(final_vs_fold_jaccard),
                "max_final_vs_fold_jaccard": np.max(final_vs_fold_jaccard),
                "final_edges_in_cv_union_fraction": len(final_edge_set & union_edges)
                / len(final_edge_set),
                "final_edges_in_all_folds_fraction": len(final_edge_set & common_edges)
                / len(final_edge_set),
            }
        )

    stability_records.append(stability_record)

In [ ]:
stability_summary = pd.DataFrame(stability_records).sort_values("graph")
final_edge_support = pd.DataFrame(final_edge_support_records)


def plot_edge_support(df, *, x_label, y_label, filename, order=None):
    distribution = (
        df.groupby(["graph", "n_folds"], as_index=False)
        .size()
        .rename(columns={"size": "n_edges"})
    )
    figure, axis = plt.subplots(figsize=(8, 4.5))
    sns.barplot(
        data=distribution,
        x="n_folds",
        y="n_edges",
        hue="graph",
        order=order,
        palette=PALETTE,
        ax=axis,
    )
    axis.set_xlabel(x_label)
    axis.set_ylabel(y_label)
    axis.legend(title="graf", frameon=False)
    figure.tight_layout()
    save_figure(figure, filename)
    plt.show()


plot_edge_support(
    edge_frequency,
    x_label="liczba foldów zawierających krawędź",
    y_label="liczba krawędzi",
    filename="evaluation__medical__graph_edge_stability",
)
plot_edge_support(
    final_edge_support,
    x_label="liczba foldów zawierających finalną krawędź",
    y_label="liczba krawędzi grafu finalnego",
    filename="evaluation__medical__final_graph_edge_support",
    order=range(EXPECTED_FOLDS + 1),
)

final_edge_ranking = final_edge_support.copy()
final_edge_ranking["edge"] = (
    final_edge_ranking["source"] + " -- " + final_edge_ranking["target"]
)
final_edge_ranking["fold_fraction"] = final_edge_ranking["n_folds"] / EXPECTED_FOLDS
final_edge_ranking["rank"] = (
    final_edge_ranking.groupby("graph")["n_folds"]
    .rank(method="dense", ascending=False)
    .astype(int)
)
final_edge_ranking = final_edge_ranking.sort_values(
    ["graph", "rank", "source", "target"]
).reset_index(drop=True)

save_table(
    stability_summary.set_index("graph").round(4),
    "evaluation__medical__graph_stability_summary",
    float_format="%.4f",
)
final_edge_ranking.to_csv(
    Path(RESULTS_DIR) / "medical_final_graph_edge_ranking.csv",
    index=False,
)
stability_summary.round(4)

In [ ]:
for graph, ranking in final_edge_ranking.groupby("graph"):
    top_edges = ranking.head(20)[
        [
            "source",
            "target",
            "edge",
            "n_folds",
        ]
    ].reset_index(drop=True)
    save_table(top_edges, f"{graph}_top_edges")
    display(top_edges)

## Exporting tables and plots

The full and rounded results tables are saved as CSV files, and the report-ready version is also exported to LaTeX. Separate AUC, PR AUC, and edge-stability plots are exported through the project's shared reporting function; the graph-stability summary is additionally saved as a LaTeX table.

In [ ]:
output_dir = Path(RESULTS_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

full_csv_path = output_dir / "medical_cv_and_final_test_metrics.csv"
rounded_csv_path = output_dir / "medical_cv_and_final_test_metrics_rounded.csv"
results_table.to_csv(full_csv_path, index=False)
results_table.round(4).to_csv(rounded_csv_path, index=False)

latex_table = results_table.drop(columns="label").set_index(["model", "graph"])
save_table(
    latex_table.round(4),
    "evaluation__medical__cv_and_final_test_metrics",
    float_format="%.4f",
)

print(f"Saved: {full_csv_path}")
print(f"Saved: {rounded_csv_path}")